# Minimising Loss

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from course_utils.plots import loss_landscape, plot_scatter_plot, mark_solution_point

## Create Example Observation and Loss Function

In the previous lesson we learnt what loss functions are and why they are useful. 

Next we need to consider systematic approaches to how we might minimise the result of our loss function in order to create a more performant model.

Let's look at an example observation of heights (cm) and weights (kg).

In [ ]:
heights_cm = np.array([
    172, 158, 184, 166, 177,
    191, 163, 180, 155, 174,
    188, 169, 182, 160, 176,
    193, 167, 185, 171, 179
], dtype=float)

weights_kg = np.array([
    74, 61, 92, 68, 83,
    106, 57, 78, 52, 89,
    84, 98, 76, 64, 71,
    112, 73, 87, 69, 95
], dtype=float)

plot_scatter_plot(
    x=heights_cm, 
    y=weights_kg, 
    x_label="Height (cm)", 
    y_label="Weight (kg)"
)

As in previous examples we will use the ***Mean Square Error (MSE)*** for our loss functon here which we define as:

In [ ]:
def mse(y_observed: np.array, y_predicted: np.array) -> float:
    return np.mean((y_observed - y_predicted) ** 2)

## Visualise Possible Loss Values

Before we look at how to find the minimum possible loss value let's assume that the model we want to fit to our observations is a single input linear regression model of the form:

$$
\operatorname{weight} = m \times \operatorname{height} + b
$$

where:

- $m$ is the gradient,
- $b$ is the intercept.

Now lets define a reasonable set of possible values for $m$ and $b$ respectively and plot them against the result of the loss function for each pair of $m, b$.

We call this our ***Loss Landscape***.

In [ ]:
possible_m_values = np.linspace(0.5, 1.8, 100)
possible_b_values = np.linspace(-180, -80, 100)

fig, ax = loss_landscape(
    x_observed=heights_cm, 
    y_observed=weights_kg, 
    possible_m_values=possible_m_values, 
    possible_b_values=possible_b_values, 
    loss_function=mse)

## Solving for Loss

We can see from the loss landscape that there exists a pair of \(m, b\) that gives us the minimum possible loss.

Let's represent this algebraically before we attempt to find out what this missing value is:

#### Representing the Loss Landscape as a Matrix Equation

The general form of our single input variable linear regression model is:

$$
\hat{y} = mx + b
$$

This means that for $n$ observations we must have:

$$
\begin{aligned}
\hat{y}_1 &= mx_1 + b \\
\hat{y}_2 &= mx_2 + b \\
&\;\vdots \\
\hat{y}_n &= mx_n + b
\end{aligned}
$$

We can represent this list of equations as 2 equivalent matrics like so:

$$
\begin{bmatrix}
\hat{y}_1 \\
\hat{y}_2 \\
\vdots \\
\hat{y}_n
\end{bmatrix}
=
\begin{bmatrix}
mx_1 + b \\
mx_2 + b \\
mx_3 + b \\
\vdots \\
mx_n + b
\end{bmatrix}
$$

Using the rules of matrix multiplication we can rewrite this as the following:

$$
\begin{bmatrix}
\hat{y}_1 \\
\hat{y}_2 \\
\vdots \\
\hat{y}_n
\end{bmatrix}
=
\begin{bmatrix}
x_1 & 1 \\
x_2 & 1 \\
x_3 & 1 \\
\vdots & \vdots \\
x_n & 1
\end{bmatrix}
\begin{bmatrix}
m \\
b
\end{bmatrix}
$$

Now we have 3 matrices that can be denoted more easily as:

$$
\hat{\mathbf{y}} = X\theta
$$

where:

- $\hat{\mathbf{y}}$ is the vector of our predictions,
- $X$ is our ***Design Matrix***,
- $\theta$ is the vector containing our gradient ($m$) and intercept ($b$). 

#### Using our Matrix Equation to Solve for Loss

Now we have an algebraic representation of our possible loss values we can find which pair of gradient ($m$) and intercept ($b$) will return the lowest possible loss. 

Our loss function here is ***Mean Squared Error***. Whatever value of $\theta$ returns the minimum ***MSE*** will also return the minimum ***Sum of Squared Residuals*** as this is $\operatorname{MSE} \times n$.

Using the above equation we can represent our residuals as the following:

$$
r = \mathbf{y} - X\theta
$$

where:

- $r$ is the ***residual vector***,
- $y$ is the vector of observed values (this is also known as ***response vector*** or ***target vector***).

There for the we can represent the ***Sum of Squared Residuals*** in terms of $\theta$ like so:
$$
\begin{aligned}
r_1^2 + r_2^2 + \cdots + r_n^2
&=
\begin{bmatrix}
r_1 & r_2 & \cdots & r_n
\end{bmatrix}
\begin{bmatrix}
r_1 \\
r_2 \\
\vdots \\
r_n
\end{bmatrix} \\
&=
\mathbf{r}^T\mathbf{r} \\
&=
(\mathbf{y}-X\theta)^T(\mathbf{y}-X\theta).
\end{aligned}
$$

This is the quantity we want to minimise. We will call it $J(\theta)$:

$$
J(\theta) = (\mathbf{y} - X\theta)^T(\mathbf{y} - X\theta)
$$

To find the minimum, we differentiate $J(\theta)$ with respect to $\theta$ and set the result equal to zero.

First expand the expression:

$$
\begin{aligned}
J(\theta)
&=
(\mathbf{y} - X\theta)^T(\mathbf{y} - X\theta) \\
&= 
(\mathbf{y}^T - \theta^T X^T)(\mathbf{y} - X\theta) \\
&=
\mathbf{y}^T\mathbf{y} - \mathbf{y}^TX\theta - \theta^TX^Ty + \theta^TX^TX\theta \\
&=
\mathbf{y}^T\mathbf{y} - 2\theta^TX^T\mathbf{y} + \theta^TX^TX\theta
\end{aligned}
$$

Now differentiate with respect to $\theta$:

$$
\frac{\partial J}{\partial \theta}
=
-2X^T\mathbf{y}
+
2X^TX\theta
$$

At the minimum, the gradient is zero:

$$
-2X^T\mathbf{y}
+
2X^TX\theta
=
0
$$

Divide through by 2:

$$
-X^T\mathbf{y}
+
X^TX\theta
=
0
$$

Rearrange:

$$
X^TX\theta
=
X^T\mathbf{y}
$$

These are called the **normal equations**.

To solve for $\theta$, multiply both sides by $(X^TX)^{-1}$:

$$
\theta
=
(X^TX)^{-1}X^T\mathbf{y}
$$

This is the **normal equation solution**.

It gives us the values of $m$ and $b$ that minimise the sum of squared residuals.

### Applying the Normal Equation to Fit a Linear Regression

Now we have derived the ***Normal Equation*** lets implement this in Python so we can use it to fit our model to our observations above. 

In [ ]:
def fit_linear_regression_normal_equation(
    X_observed: np.ndarray,
    y_observed: np.ndarray,
) -> np.ndarray:
    # Add the intercept column to create the desing matrix
    X = np.column_stack([X_observed, np.ones(len(X_observed))])

    theta = np.linalg.inv(X.T @ X) @ X.T @ y_observed
    return theta

Next we use our observations as inputs to the normal equation to find the optimum gradient ($m$) and intercept ($b$):

In [ ]:
theta = fit_linear_regression_normal_equation(
    X_observed=heights_cm, 
    y_observed=weights_kg
)

m = theta[0]
b = theta[1]

weights_predicted = m * heights_cm + b
loss = mse(weights_kg, weights_predicted)

print(f"The gradient that minimises loss for our linear regression is: {m:.2f}")
print(f"The intercept that minimises loss for our linear regression is: {b:.2f}")
print(f"This fitted linear regression has a loss of {loss:.2f}")

### Marking the Solution on the Loss Landscape

Now lets mark the solution on our loss landscape:

In [ ]:
fig, ax = loss_landscape(
    x_observed=heights_cm,
    y_observed=weights_kg,
    possible_m_values=possible_m_values,
    possible_b_values=possible_b_values,
    loss_function=mse
)

mark_solution_point(ax, m, b, loss)

### Plotting the Fitted Linear Regression

Finally, we can plot our fitted linear regression against our observations to see how it fits:

In [ ]:
# Heights covering the full plotted range
line_heights = np.linspace(
    heights_cm.min(),
    heights_cm.max(),
    100
)

# Weights predicted across our full plotted range of heights
line_weights = m * line_heights + b

plot_scatter_plot(
    x=heights_cm, 
    y=weights_kg, 
    x_label="Height (cm)", 
    y_label="Weight (kg)",
    line_values=(line_heights, line_weights)
)


## Using the Normal Equation to Fit Multiple Input Linear Regression Models

As covered in the first lesson the general equation for a linear regression is the following:

$$
\hat{y} = w_1x + w_2x + \ldots + w_nx + b
$$

where:
- each $w_i$ is a weight,
- $b$ is the bias.

and the ***Normal Equation*** can be used to find the values of $w_1, w_2, \ldots, w_n$ and $b$ that minimise the ***Mean Squared Error*** for all $n \in \mathbb{N}$.

### Multiple Input Linear Regression Example

Let's return an example of a multiple input Linear Regresson from the first lesson. This example contains 20 observations, and for each observation we have their height (cm), weight (kg) and age (years).

In [ ]:
heights_cm = np.array([
    172, 158, 184, 166, 177,
    191, 163, 180, 155, 174,
    188, 169, 182, 160, 176,
    193, 167, 185, 171, 179
], dtype=float)

age_yrs = np.array([
    34, 52, 28, 41, 37,
    24, 59, 46, 31, 63,
    27, 55, 39, 22, 48,
    35, 44, 61, 29, 50
], dtype=float)

weights_kg = np.array([
    74, 61, 92, 68, 83,
    106, 57, 78, 52, 89,
    84, 98, 76, 64, 71,
    112, 73, 87, 69, 95
], dtype=float)

plot_scatter_plot(
    x=[heights_cm, age_yrs], 
    y=weights_kg, 
    x_label=["Height (cm)", "Age (years)"], 
    y_label="Weight (kg)"
)

Unlike the single input linear regression example above we cannot plot the loss landscape as that would require four dimensions ($w_1, w_2, b$ and loss).

### Fitting a Multiple Input Linear Regression

Now we can use the Normal Equation to fit our multiple input linear regression to our observations.

In [ ]:
# Create the observed feature matrix
X_observed = np.column_stack([
    heights_cm,
    age_yrs,
])

# Fit the linear regression
theta = fit_linear_regression_normal_equation(
    X_observed=X_observed,
    y_observed=weights_kg,
)

# Extract the fitted parameters
height_weight = theta[0]
age_weight = theta[1]
intercept = theta[2]

# Make predictions
weights_predicted = (
    height_weight * heights_cm
    + age_weight * age_yrs
    + intercept
)

# Measure the loss
loss = mse(weights_kg, weights_predicted)

print(f"Height weight: {height_weight:.2f}")
print(f"Age weight: {age_weight:.2f}")
print(f"Intercept: {intercept:.2f}")
print(f"This fitted linear regression has a loss of {loss:.2f}")

### Plotting the Fitted Multiple Input Linear Regression

Finally, we can plot our fitted linear regression against our observations to see how it fits:

In [ ]:
# Heights and ages covering the full plotted range
surface_heights, surface_ages = np.meshgrid(
    np.linspace(heights_cm.min(), heights_cm.max(), 30),
    np.linspace(age_yrs.min(), age_yrs.max(), 30)
)

# Weights predicted across our full plotted range of heights and ages
surface_weights = (
    height_weight * surface_heights
    + age_weight * surface_ages
    + intercept
    )

plot_scatter_plot(
    x=[heights_cm, age_yrs],
    y=weights_kg,
    x_label=["Height (cm)", "Age (years)"],
    y_label="Weight (kg)",
    line_values=(surface_heights, surface_ages, surface_weights)
)